# EMA Regime-Hold Multi-Exchange Sweep

**Multi-timeframe** optimization: requires BOTH signal_interval
and regime_interval candles per pair. Pairs missing either are skipped.

This notebook:
1. Discovers all pairs that have both intervals available
2. For each pair, loads both streams, audits both, runs Optuna
3. Exports YAML under `artifacts/direction-custom/ema_regime_hold/<connector>/`

**Configuration:** Edit the variables in the first code cell, then Run All.


In [1]:
import sys, os, subprocess, time, logging
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")
print(f"Strategy: ema_regime_hold")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")


pmm_lab 0.2.0 | NumPy 2.2.6 | Optuna 4.7.0
Strategy: ema_regime_hold
MONGO_URI      : SET
OPTUNA_STORAGE : SET


## 1. Configuration


In [2]:
# ==============================================================
# EMA REGIME-HOLD MULTI-EXCHANGE SWEEP CONFIGURATION
# ==============================================================
# EMA is multi-timeframe — signal_interval for fast bars + regime_interval
# for slow (trend/regime) detection. Pairs must have BOTH intervals in Mongo.
# Default is None because regime shifts ARE the signal — capping history
# would discard useful information. D4 forces hold_mode='reentry'.
# ==============================================================

CONNECTORS = ["mexc", "nonkyc"]
QUOTE_ASSET = "*"
N_TRIALS = 500
PERC_TRIALS_TEST = 0.05
TOP_N = 100
MIN_ROBUST_SCORE = -5.0
N_JOBS = 8

# EMA is multi-timeframe
SIGNAL_CONNECTOR_INTERVALS = {"nonkyc": "5m", "mexc": "5m"}
REGIME_CONNECTOR_INTERVALS = {"nonkyc": "4h", "mexc": "4h"}
DEFAULT_SIGNAL_INTERVAL = "5m"
DEFAULT_REGIME_INTERVAL = "4h"

MIN_DATA_DAYS = 120                  # EMA needs more history for 4h regime warmup
MAX_STALE_DAYS = 7
MAX_TRAINING_DAYS = None             # None = use all; regime shifts ARE the signal

SEARCH_CONTROLLER_COMPAT = False
VALIDATION_CONTROLLER_COMPAT = True
PHASE2_CONTROLLER_COMPAT = True

REFRESH_CLOSE_MODE = "market_close"
INITIAL_BASE_BALANCE = 0.0

TAKER_PROBABILITY_BY_CONNECTOR = {"nonkyc": 0.10, "mexc": 0.0}
DEFAULT_TAKER_PROBABILITY = 0.0

MIN_PHASE1_BEST_FOR_STRESS = -0.5
OBJECTIVE_VERSION = 2

RECENT_BLOCKING_WINDOW_DAYS = 28
RECENT_INFORMATIONAL_WINDOW_DAYS = [14, 7]
RECENT_REPORT_WINDOW_DAYS = sorted(
    dict.fromkeys([RECENT_BLOCKING_WINDOW_DAYS] + RECENT_INFORMATIONAL_WINDOW_DAYS),
    reverse=True,
)

CONNECTORS = [c.strip().lower() for c in CONNECTORS]

from pmm_lab.config.defaults import INTERVAL_SECONDS

print(f"Strategy       : ema_regime_hold_v1")
print(f"Connectors     : {', '.join(CONNECTORS)}")
print(f"Signal intvls  : {SIGNAL_CONNECTOR_INTERVALS}")
print(f"Regime intvls  : {REGIME_CONNECTOR_INTERVALS}")
print(f"Trials/pair    : {N_TRIALS}")
print(f"Min data days  : {MIN_DATA_DAYS}")
print(f"Max training   : {MAX_TRAINING_DAYS if MAX_TRAINING_DAYS else 'unlimited'}")
print(f"Max stale days : {MAX_STALE_DAYS}")


Strategy       : ema_regime_hold_v1
Connectors     : mexc, nonkyc
Signal intvls  : {'nonkyc': '5m', 'mexc': '5m'}
Regime intvls  : {'nonkyc': '4h', 'mexc': '4h'}
Trials/pair    : 500
Min data days  : 120
Max training   : unlimited
Max stale days : 7


In [3]:
# ── Preflight: validate storage + worker configuration ──
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

print(f"Requested N_JOBS: {N_JOBS}")
print(f"Storage backend : {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")
print(f"Dispatch mode   : {'process-parallel' if N_JOBS > 1 and _is_postgres else 'serial'}")
if N_JOBS > 1 and not _is_postgres:
    print("WARNING: N_JOBS>1 with SQLite — forcing serial. Set OPTUNA_STORAGE for parallelism.")


Requested N_JOBS: 8
Storage backend : PostgreSQL
Dispatch mode   : process-parallel


## 2. Discover Available Pairs Across Exchanges

In [4]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=None, quote_asset=QUOTE_ASSET)

now_ts = datetime.now(timezone.utc).timestamp()

# EMA needs BOTH signal_interval AND regime_interval candles per (connector, pair).
pair_groups = {}
for combo in all_combos:
    connector = combo["connector"]
    if connector not in CONNECTORS:
        continue
    signal_i = SIGNAL_CONNECTOR_INTERVALS.get(connector, DEFAULT_SIGNAL_INTERVAL)
    regime_i = REGIME_CONNECTOR_INTERVALS.get(connector, DEFAULT_REGIME_INTERVAL)
    if combo["interval"] not in (signal_i, regime_i):
        continue
    key = (connector, combo["trading_pair"])
    pair_groups.setdefault(key, {})[combo["interval"]] = combo

candidates = []
stale_exclusions = []
insufficient_exclusions = []
missing_interval_exclusions = []

for (connector, pair), ivs in pair_groups.items():
    signal_i = SIGNAL_CONNECTOR_INTERVALS.get(connector, DEFAULT_SIGNAL_INTERVAL)
    regime_i = REGIME_CONNECTOR_INTERVALS.get(connector, DEFAULT_REGIME_INTERVAL)
    if signal_i not in ivs or regime_i not in ivs:
        missing_interval_exclusions.append({
            "connector": connector, "trading_pair": pair,
            "present": sorted(ivs.keys()),
            "reason": f"missing {signal_i}" if signal_i not in ivs else f"missing {regime_i}",
        })
        continue
    s = ivs[signal_i]
    r = ivs[regime_i]

    effective_signal_first = s["first_ts"]
    effective_regime_first = r["first_ts"]
    if MAX_TRAINING_DAYS is not None:
        cutoff = min(s["last_ts"], r["last_ts"]) - (MAX_TRAINING_DAYS * 86400)
        effective_signal_first = max(effective_signal_first, cutoff)
        effective_regime_first = max(effective_regime_first, cutoff)

    effective_first_ts = max(effective_signal_first, effective_regime_first)
    effective_last_ts = min(s["last_ts"], r["last_ts"])
    data_days = (effective_last_ts - effective_first_ts) / 86400

    if data_days < MIN_DATA_DAYS:
        insufficient_exclusions.append({
            "connector": connector, "trading_pair": pair,
            "data_days": data_days,
            "reason": f"< {MIN_DATA_DAYS}d cross-interval data",
        })
        continue

    last_age_days = (now_ts - effective_last_ts) / 86400
    if last_age_days > MAX_STALE_DAYS:
        stale_exclusions.append({
            "connector": connector, "trading_pair": pair,
            "last_age_days": last_age_days,
            "reason": f"stale ({last_age_days:.1f}d old > {MAX_STALE_DAYS}d)",
        })
        continue

    candidates.append({
        "connector": connector, "trading_pair": pair,
        "signal_interval": signal_i, "regime_interval": regime_i,
        "signal_first_ts": effective_signal_first,
        "regime_first_ts": effective_regime_first,
        "effective_first_ts": effective_first_ts,
        "signal_last_ts": s["last_ts"],
        "regime_last_ts": r["last_ts"],
        "signal_count": s["count"],
        "regime_count": r["count"],
        "data_days": data_days,
    })

candidates = sorted(candidates, key=lambda c: (c["connector"], c["trading_pair"]))

print(f"\n{'='*60}")
print(f"Found {len(candidates)} connector/pair combos with >= {MIN_DATA_DAYS}d of BOTH intervals")
print(f"{'='*60}")

for connector in CONNECTORS:
    s = [c for c in candidates if c["connector"] == connector]
    signal_i = SIGNAL_CONNECTOR_INTERVALS.get(connector, DEFAULT_SIGNAL_INTERVAL)
    regime_i = REGIME_CONNECTOR_INTERVALS.get(connector, DEFAULT_REGIME_INTERVAL)
    print(f"\n{connector} / {signal_i}+{regime_i}: {len(s)} pair(s)")
    for c in s:
        print(f"  {c['trading_pair']:15s} {c['data_days']:6.1f}d  "
              f"signal={c['signal_count']:>7,} regime={c['regime_count']:>5,}")

if missing_interval_exclusions:
    print(f"\nExcluded {len(missing_interval_exclusions)} pair(s) missing an interval:")
    for ex in missing_interval_exclusions[:10]:
        print(f"  {ex['connector']:8s} {ex['trading_pair']:15s} {ex['reason']}")

if stale_exclusions:
    print(f"\nExcluded {len(stale_exclusions)} stale pair(s)")
if insufficient_exclusions:
    print(f"\nExcluded {len(insufficient_exclusions)} pair(s) with insufficient data")

print(f"\nTotal to optimize: {len(candidates)}")



Found 77 connector/pair combos with >= 120d of BOTH intervals

mexc / 5m+4h: 32 pair(s)
  ADA-USDT         204.2d  signal=103,804 regime=1,226
  APT-USDT         220.7d  signal=103,804 regime=1,325
  ASTER-USDT       204.1d  signal= 58,894 regime=1,226
  ATOM-USDT        204.2d  signal=103,803 regime=1,226
  BNB-USDT         220.7d  signal=103,802 regime=1,325
  BTC-USDT         360.0d  signal=103,804 regime=17,261
  DOGE-USDT        360.0d  signal=103,805 regime=16,194
  DOT-USDT         204.2d  signal=103,802 regime=1,226
  ETH-USDT         360.0d  signal=103,810 regime=17,261
  FET-USDT         193.3d  signal=103,803 regime=1,161
  HYPE-USDT        204.2d  signal=103,802 regime=1,226
  ICP-USDT         204.2d  signal=103,802 regime=1,226
  LTC-USDT         360.0d  signal=103,801 regime=17,261
  OP-USDT          204.2d  signal=103,800 regime=1,226
  PEPE-USDT        220.7d  signal=103,800 regime=1,325
  PUMP-USDT        204.1d  signal= 58,889 regime=1,226
  RENDER-USDT      204.2d  

## 3. Sweep: Optimize Each Connector / Pair

For each pair, loads both signal and regime candles, audits both,
and runs the EMA regime-hold objective.


In [5]:
# ── Config guard ──
_required_config = [
    "VALIDATION_CONTROLLER_COMPAT", "SEARCH_CONTROLLER_COMPAT", "PHASE2_CONTROLLER_COMPAT",
    "OBJECTIVE_VERSION", "N_TRIALS", "TOP_N", "MIN_ROBUST_SCORE", "N_JOBS",
]
_missing = [v for v in _required_config if v not in globals()]
if _missing:
    raise RuntimeError(f"Config cell not executed; missing: {_missing}")

from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.canonicalizer_ema_regime_hold import canonicalize_ema_regime_hold_params
from pmm_lab.optuna.search_space_ema_regime_hold import suggest_ema_regime_hold_params
from pmm_lab.export.hb_yaml_ema_regime_hold import (
    EMARegimeHoldExportParams, export_ema_regime_hold_yaml, validate_export_ema_regime_hold,
)
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.objective import REJECT_SCORE
import time

stress_scenarios = load_stress_scenarios()
rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

for pair_idx, pair_info in enumerate(candidates):
    connector = pair_info["connector"]
    pair = pair_info["trading_pair"]
    signal_interval = pair_info["signal_interval"]
    regime_interval = pair_info["regime_interval"]
    bar_interval_seconds = INTERVAL_SECONDS[signal_interval]

    print(f"\n{'='*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {connector} / {pair} / {signal_interval}+{regime_interval}")
    print(f"{'='*60}")

    pair_start = time.time()

    # ── Load both candle streams ──
    try:
        _signal_start = int(pair_info.get("signal_first_ts")) if MAX_TRAINING_DAYS is not None else None
        _regime_start = int(pair_info.get("regime_first_ts")) if MAX_TRAINING_DAYS is not None else None
        signal_candles = loader.load_range(
            DataQuery(connector=connector, trading_pair=pair, interval=signal_interval, start_ts=_signal_start)
        )
        regime_candles = loader.load_range(
            DataQuery(connector=connector, trading_pair=pair, interval=regime_interval, start_ts=_regime_start)
        )
    except Exception as e:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "load_error", "error": str(e)})
        continue

    # ── Audit both ──
    signal_audit = validate_candles(signal_candles, interval=signal_interval, strict=True)
    if not signal_audit.passed_strict:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "audit_fail_signal",
                              "reasons": signal_audit.failure_reasons})
        continue
    regime_audit = validate_candles(regime_candles, interval=regime_interval, strict=True)
    if not regime_audit.passed_strict:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "audit_fail_regime",
                              "reasons": regime_audit.failure_reasons})
        continue

    dataset_hash = hash_candles(signal_candles)
    reference_price = float(signal_candles["close"][-1])
    pair_rules = resolve_pair_rules(rules_db, connector, pair)
    taker_prob = TAKER_PROBABILITY_BY_CONNECTOR.get(connector, DEFAULT_TAKER_PROBABILITY)

    try:
        objective = create_objective(
            candles=signal_candles, pair_rules=pair_rules,
            bar_interval_seconds=bar_interval_seconds,
            dataset_hash=dataset_hash, reference_price=reference_price,
            strategy_name="ema_regime_hold",
            objective_version=OBJECTIVE_VERSION,
            run_stress=False,
            controller_compat=SEARCH_CONTROLLER_COMPAT,
            refresh_close_mode=REFRESH_CLOSE_MODE,
            initial_base_balance=INITIAL_BASE_BALANCE,
            taker_probability=taker_prob,
            regime_candles=regime_candles,
        )
    except Exception as e:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "objective_error", "error": str(e)})
        continue

    import optuna
    study_name = f"ema_regime_hold_{connector}_{pair.replace('-', '_').lower()}"
    study = optuna.create_study(direction="maximize", study_name=study_name, load_if_exists=False)
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, catch=(Exception,))

    completed = [t for t in study.trials
                 if t.state == optuna.trial.TrialState.COMPLETE
                 and t.user_attrs.get("reject_reason") is None]
    if not completed:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "no_valid_trials"})
        continue

    completed.sort(key=lambda t: t.user_attrs.get("objective_score", REJECT_SCORE), reverse=True)
    best = completed[0]
    best_score = float(best.user_attrs.get("objective_score", REJECT_SCORE))

    if best_score < MIN_PHASE1_BEST_FOR_STRESS:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "below_phase1_gate",
                              "best_score": best_score})
        continue

    raw = dict(best.params)
    raw.setdefault("hold_mode", "reentry")          # D4
    raw.setdefault("max_executors_per_side", 1)
    raw.setdefault("total_amount_quote", 300.0)

    bundle, reason = canonicalize_ema_regime_hold_params(
        raw, pair_rules, reference_price,
        signal_interval_seconds=bar_interval_seconds,
        regime_candles=regime_candles,
    )
    if bundle is None:
        sweep_results.append({"connector": connector, "trading_pair": pair, "status": "canonicalize_reject",
                              "reason": reason})
        continue

    out_dir = Path("artifacts/direction-custom/ema_regime_hold") / connector
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{connector}_{pair.replace('-', '_').lower()}_ema_regime_hold_v1.yml"
    export_params = EMARegimeHoldExportParams(
        connector_name=connector, trading_pair=pair,
        signal_interval=signal_interval, regime_interval=regime_interval,
    )
    export_ema_regime_hold_yaml(bundle.strategy_config, bundle.engine_config, export_params, out_path)
    validate_export_ema_regime_hold(out_path)

    sweep_results.append({
        "connector": connector, "trading_pair": pair,
        "signal_interval": signal_interval, "regime_interval": regime_interval,
        "status": "complete",
        "best_score": best_score,
        "yaml_path": str(out_path),
        "n_trials_completed": len(completed),
        "elapsed_s": time.time() - pair_start,
    })

print(f"\nTotal sweep time: {time.time() - sweep_start:.1f}s")



  [1/77] mexc / ADA-USDT / 5m+4h


Bar 2432: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 938: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0


  [2/77] mexc / APT-USDT / 5m+4h


Bar 2756: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 1665: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 


  [3/77] mexc / ASTER-USDT / 5m+4h


Bar 1878: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 1193: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 0: no orders placed (this warning will not repeat)
Bar 

KeyboardInterrupt: 

## 4. Results Summary

In [ ]:
# Results summary — exclusion stats and per-pair outcome table.
from pathlib import Path

def _status_counts(rows):
    counts = {}
    for r in rows:
        counts[r.get("status", "?")] = counts.get(r.get("status", "?"), 0) + 1
    return counts

print("=" * 60)
print("SWEEP RESULTS SUMMARY")
print("=" * 60)
print("Status counts:", _status_counts(sweep_results))

print("\nPer-pair outcomes:")
for r in sweep_results:
    status = r.get("status", "?")
    conn = r.get("connector", "?")
    pair = r.get("trading_pair", "?")
    extras = ""
    if status == "complete":
        extras = f" score={r.get('best_score', 0):.3f}  yaml={r.get('yaml_path')}"
    elif "reason" in r:
        extras = f" reason={r['reason']}"
    elif "error" in r:
        extras = f" error={r['error'][:80]}"
    print(f"  [{status:20s}] {conn:8s} {pair:15s}{extras}")


## 5. Profitable Pairs Detail by Exchange

In [ ]:
profitable = [r for r in sweep_results if r["status"] == "complete" and r.get("best_score", 0) > 0]
print(f"\n{'='*60}")
print(f"Profitable pairs: {len(profitable)}")
print(f"{'='*60}")

print("\nRelease Gates (Informational Only):")
for r in profitable:
    print(f"\n  {r['connector']} / {r['trading_pair']}:")
    gates = [
        ("robust_score > 0", r.get("best_score", 0), 0.0,
         r.get("best_score", 0) > 0),
    ]
    for name, actual, threshold, passed in gates:
        mark = "PASS" if passed else "FAIL"
        print(f"    [{mark}] {name}: actual={actual}")


## 6. Next Steps

- Inspect the per-pair markdown reports under `artifacts/direction-custom/ema_regime_hold/<connector>/`.
- Review exported YAMLs against the live Hummingbot controller Pydantic model.
- For finalists, run the retest notebook with a narrowed `RETEST_PAIRS` list.
- All release gates are informational only per the user's directive; only
  the strict data-audit gate hard-stops (per pair — a failed audit `continue`s
  to the next pair, not halting the whole notebook).
